# TDo2 — Controlled RL evaluation and safety release — Core

**Student starter — complete the TODO cells**  
**Scope:** Time Series and Reinforcement Learning; CPU only

## Working and claim contract

Local Python 3.11/3.12 with the locked `uv` environment is the primary path. Colab availability is governed by the release evidence for the exact archive in use. Public base seeds are `11, 23, 37, 51, 71`. TD1–TD4 claim rows are **provisional development evidence**: they support method choice and provenance but cannot count as capstone empirical release claims. Only the protected/controlled evidence produced in TDo1 and TDo2 may support final empirical capstone claims. Restart and run top-to-bottom before handoff.

## Learning contract

The less-scaffolded Core route assesses the same outcomes and evidence standard as Support.

In [1]:
from pathlib import Path
import os, sys

def locate_course_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate/'uv.lock').is_file() and (candidate/'shared').is_dir(): return candidate.resolve()
    for lock in start.glob('*/uv.lock'):
        if (lock.parent/'shared').is_dir(): return lock.parent.resolve()
    raise FileNotFoundError('Extract the complete course bundle and run from inside it.')
COURSE_ROOT=locate_course_root(); os.chdir(COURSE_ROOT)
if str(COURSE_ROOT) not in sys.path: sys.path.insert(0,str(COURSE_ROOT))
import shared
print('Course root:',COURSE_ROOT)
print('Primary path: locked local CPU; check the release evidence before using Colab')

Course root: C:\Users\tjoli\Downloads\Advanced_Machine_Learning_ESILV_2026-2027_DVL_V1.1 (1)
Primary path: locked local CPU; check the release evidence before using Colab


## 1 — Frozen table and provenance

In [2]:
import hashlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from shared.data import PUBLIC_SEEDS
from shared.evaluation import mean_ci95
from shared.rl import evaluate_policies,q_table_policy,random_policy,reorder_policy
q_path=COURSE_ROOT/'02_labs/TD4_tabular_q_learning/q_table.npy'
q_table=np.load(q_path,allow_pickle=False) if q_path.is_file() else None
if q_table is None: print('Place frozen TD4 q_table.npy at',q_path)
else:
    assert q_table.shape==(9,2,3) and np.isfinite(q_table).all()
    print('Q-table SHA256:',hashlib.sha256(q_path.read_bytes()).hexdigest())

Q-table SHA256: 61cce1915bd5798f53c22529c9dbdca52b38c2c946a5b186e28e2d777f19738e


## 2 — Public safety and injected evaluator

In [3]:
safety_seeds=tuple(seed+200000 for seed in PUBLIC_SEEDS); public_safety=pd.DataFrame()
def compare_on_injected_seeds(evaluation_seeds):
    evaluation_seeds=tuple(int(seed) for seed in evaluation_seeds)
    assert evaluation_seeds and len(set(evaluation_seeds))==len(evaluation_seeds)
    if q_table is None: return None
    policies={'random':random_policy,'reorder':reorder_policy,'q_learning':q_table_policy(q_table)}
    comparison=evaluate_policies(policies,seeds=evaluation_seeds)
    assert comparison.groupby('policy').seed.nunique().eq(len(evaluation_seeds)).all()
    return comparison

if q_table is not None:
    policies={'random':random_policy,'reorder':reorder_policy,'q_learning':q_table_policy(q_table)}
    public_safety=evaluate_policies(policies,seeds=safety_seeds)
    assert len(public_safety)==15 and public_safety.groupby('policy').seed.nunique().eq(5).all()
public_safety

,seed,policy,episode_cost,mean_fill_rate,unmet_demand,mean_stock,mean_order_quantity
0,200011,random,88.95,0.808333,16,1.875,1.025
1,200011,reorder,40.20,0.983333,2,0.825,1.350
2,200011,q_learning,39.60,0.983333,2,0.775,1.350
3,200023,random,91.95,0.854167,13,2.575,1.225
4,200023,reorder,44.25,0.975000,3,0.800,1.425
5,200023,q_learning,45.60,0.966667,4,0.700,1.400
6,200037,random,70.50,0.925000,6,2.800,1.050
7,200037,reorder,37.20,0.991667,1,1.125,1.150
8,200037,q_learning,34.80,0.991667,1,0.925,1.150
9,200051,random,58.35,0.979167,2,2.675,1.125


## 3 — Controlled aggregate and release

Verify receipt/hashes; interpret Student-t cost/fill intervals plus unmet/stock/orders; apply the predeclared rule.

In [4]:
def summarize_safety_intervals(df):
    rows=[]
    for pol,grp in df.groupby('policy'):
        c_m,c_l,c_h=mean_ci95(grp['episode_cost'])
        f_m,f_l,f_h=mean_ci95(grp['mean_fill_rate'])
        rows.append({
            'policy':pol,'episodes':len(grp),
            'cost_mean':round(c_m,3),'cost_ci95_low':round(c_l,3),'cost_ci95_high':round(c_h,3),
            'fill_mean':round(f_m,4),'fill_ci95_low':round(f_l,4),'fill_ci95_high':round(f_h,4),
            'unmet_mean':round(float(grp['unmet_demand'].mean()),3),
            'stock_mean':round(float(grp['mean_stock'].mean()),3),
            'order_mean':round(float(grp['mean_order_quantity'].mean()),3),
        })
    return pd.DataFrame(rows)

def evaluate_release_gate(summary_df):
    q_row=summary_df.loc[summary_df['policy']=='q_learning'].iloc[0]
    r_row=summary_df.loc[summary_df['policy']=='reorder'].iloc[0]
    service_ok=bool(q_row['fill_mean']>=0.98)
    cost_bound_ok=bool(q_row['cost_mean']<=1.05*r_row['cost_mean'])
    cost_improves=bool(q_row['cost_mean']<r_row['cost_mean'])
    if service_ok and cost_improves: return 'accept'
    if service_ok and cost_bound_ok: return 'conditional'
    return 'reject'

public_summary=summarize_safety_intervals(public_safety)
print('Public safety replay (seeds 200011..200071) diagnostic gate:',evaluate_release_gate(public_summary))
print(public_summary)

agg_candidates=[
    COURSE_ROOT/'ML-Capstone-Project-Joliot-Juillard/evidence/controlled/rl_hidden_evaluation_aggregate.csv',
    COURSE_ROOT/'02_labs/TDo2_rl_safety_release/rl_hidden_evaluation_aggregate.csv',
]
controlled_aggregate=pd.DataFrame()
for cand in agg_candidates:
    if cand.is_file():
        controlled_aggregate=pd.read_csv(cand)
        break

risk_register=[
    'Sub-optimal boundary action at (stock=2, regime=1) due to finite 1750-episode exploration.',
    'Non-zero unmet demand tail risk (1.80 units/episode on public safety seeds) requires safety stock override.',
    'Marginal Student-t CIs over n=5 episodes assume approximate normality and do not test paired superiority.',
    'Markov 2-regime inventory dynamics remain disconnected from TDo1 demand forecasts (no end-to-end test).'
]

if controlled_aggregate.empty:
    release_decision='pending_controlled_receipt'
    print('Controlled evidence pending; public safety diagnostic gate =',evaluate_release_gate(public_summary))
else:
    release_decision=evaluate_release_gate(controlled_aggregate)
    print('Controlled release decision:',release_decision)

Public safety replay (seeds 200011..200071) diagnostic gate: accept
       policy  episodes  cost_mean  cost_ci95_low  cost_ci95_high  fill_mean  \
0  q_learning         5      38.22         32.489          43.951     0.9850   
1      random         5      75.33         57.289          93.371     0.8883   
2     reorder         5      39.03         34.981          43.079     0.9867   

   fill_ci95_low  fill_ci95_high  unmet_mean  stock_mean  order_mean  
0         0.9715          0.9985         1.8       0.905       1.220  
1         0.8066          0.9701         9.0       2.415       1.075  
2         0.9774          0.9959         1.6       1.015       1.225  
Controlled evidence pending; public safety diagnostic gate = accept


## 4 — Standardized individual checkpoint (20 min + 5 min submission)

In [5]:
from shared.rl import q_learning_target
def individual_transition_audit(reward,next_best,gamma,terminated,truncated,reported_cost):
    target=q_learning_target(reward,next_best,gamma,terminated=terminated)
    return {
        'target':float(target),
        'reward_equals_negative_cost':bool(np.isclose(reward,-reported_cost)),
        'bootstrapped':not bool(terminated),
        'terminated':bool(terminated),
        'truncated':bool(truncated),
    }
checkpoint_explanation='Time-limit truncation (terminated=False, truncated=True) still bootstraps future Q-value because the episode continues beyond the fixed horizon, whereas only genuine termination zeroes continuation and reward must equal -cost.'
print(individual_transition_audit(-2.,4.,.9,False,True,2.),checkpoint_explanation)

{'target': 1.6, 'reward_equals_negative_cost': True, 'bootstrapped': True, 'terminated': False, 'truncated': True} Time-limit truncation (terminated=False, truncated=True) still bootstraps future Q-value because the episode continues beyond the fixed horizon, whereas only genuine termination zeroes continuation and reward must equal -cost.


## Required interpretation

State what the evidence supports, what it does not support, the operational consequence, and one material failure mode.

In [6]:
interpretation = (
    "Across five separate public safety episodes (seeds 200011..200071), the frozen (9,2,3) Q-table (SHA256 61cce191...) "
    "cuts mean episode cost by 49.3% versus random (38.22 [95% CI 32.49, 43.95] vs 75.33), closely matches reorder cost "
    "(39.03 [95% CI 34.98, 43.08]), and satisfies the predeclared >=0.98 fill-rate floor (0.9850 vs 0.9867 for reorder). "
    "However, these five public safety episodes do not substitute for the pending instructor-controlled hidden evaluation, and "
    "overlapping marginal Student-t intervals over n=5 episodes rely on approximate normality without establishing a paired test. "
    "Operationally, release status remains pending_controlled_receipt until instructor aggregates arrive; if controlled batches fail "
    "the >=0.98 service floor or <=105% reorder-cost gate, deployment falls back immediately to the transparent reorder heuristic. "
    "A material failure mode is persistent residual unmet demand (1.80 units/episode on public safety seeds) combined with the "
    "discrete two-regime Markov assumption disconnected from TDo1 demand forecasts."
)
print(interpretation)

Across five separate public safety episodes (seeds 200011..200071), the frozen (9,2,3) Q-table (SHA256 61cce191...) cuts mean episode cost by 49.3% versus random (38.22 [95% CI 32.49, 43.95] vs 75.33), closely matches reorder cost (39.03 [95% CI 34.98, 43.08]), and satisfies the predeclared >=0.98 fill-rate floor (0.9850 vs 0.9867 for reorder). However, these five public safety episodes do not substitute for the pending instructor-controlled hidden evaluation, and overlapping marginal Student-t intervals over n=5 episodes rely on approximate normality without establishing a paired test. Operationally, release status remains pending_controlled_receipt until instructor aggregates arrive; if controlled batches fail the >=0.98 service floor or <=105% reorder-cost gate, deployment falls back immediately to the transparent reorder heuristic. A material failure mode is persistent residual unmet demand (1.80 units/episode on public safety seeds) combined with the discrete two-regime Markov ass

## Optional Extension — ungraded

Freeze valid Core first. The bounded README extension adds no points and cannot repair invalid Core evidence.